Imports

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.data import load_raw, validate_schema, create_features
from src.features import select_features_by_iv, build_woe_tables, transform_woe
from src.models import train_all_models, evaluate_models, save_model
from src.metrics import auc_roc, costo_total, build_scorecard

Carga y validación

In [2]:
df = load_raw("../data/raw/bankloan.csv")

validate_schema(df)

df = create_features(df)

df.head()

,customer,Age,Education,Employ,Address,Income,Leverage,Creddebt,OthDebt,MonthlyLoad,Default,OthDebtRatio
0,10012,28,Med,7,2.0,44.0,17.7,2.99,4.80,0.58,0,0.109091
1,10017,64,Posg,34,17.0,116.0,14.7,5.05,12.00,0.27,0,0.103448
2,10030,40,Bas,20,12.0,61.0,4.8,1.04,1.89,0.13,0,0.030984
3,10039,30,Bas,11,3.0,27.0,34.5,1.75,7.56,1.62,0,0.280000
4,10069,25,Bas,2,2.0,30.0,22.4,0.76,5.96,0.97,1,0.198667


Split y selección de features por IV

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=["Default"]),
    df["Default"],
    test_size=0.2,
    random_state=42,
    stratify=df["Default"]
)

features_sel = select_features_by_iv(
    X_train.assign(Default=y_train),
    target="Default",
    threshold=0.1
)

print(f"Features seleccionadas (IV >= 0.1): {features_sel}")

Features seleccionadas (IV >= 0.1): ['Age', 'Employ', 'Address', 'Income', 'Leverage', 'Creddebt', 'MonthlyLoad', 'OthDebtRatio']


c:\Users\Asus\OneDrive\Documentos\Magíster USACH\3 semestre\Machine Learning en Finanzas\clase_01\ml_finance_teixeiradossantos\src\features.py:26: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = temp_df.groupby("bin")[target].agg(["count", "sum"])
c:\Users\Asus\OneDrive\Documentos\Magíster USACH\3 semestre\Machine Learning en Finanzas\clase_01\ml_finance_teixeiradossantos\src\features.py:26: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = temp_df.groupby("bin")[target].agg(["count", "sum"])
c:\Users\Asus\OneDrive\Documentos\Magíster USACH\3 semestre\Machine Learning en Finanzas\clase_01\ml_fin

Transformación WoE

In [4]:
woe_tables = build_woe_tables(
    X_train.assign(Default=y_train),
    features_sel,
    target="Default"
)

X_train_woe = transform_woe(X_train[features_sel], woe_tables)
X_test_woe = transform_woe(X_test[features_sel], woe_tables)


X_train_woe.isna().sum(), X_test_woe.isna().sum()

X_train_woe.head()

c:\Users\Asus\OneDrive\Documentos\Magíster USACH\3 semestre\Machine Learning en Finanzas\clase_01\ml_finance_teixeiradossantos\src\features.py:26: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = temp_df.groupby("bin")[target].agg(["count", "sum"])
c:\Users\Asus\OneDrive\Documentos\Magíster USACH\3 semestre\Machine Learning en Finanzas\clase_01\ml_finance_teixeiradossantos\src\features.py:26: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = temp_df.groupby("bin")[target].agg(["count", "sum"])
c:\Users\Asus\OneDrive\Documentos\Magíster USACH\3 semestre\Machine Learning en Finanzas\clase_01\ml_fin

,Age_woe,Employ_woe,Address_woe,Income_woe,Leverage_woe,Creddebt_woe,MonthlyLoad_woe,OthDebtRatio_woe
1110,-0.066344,0.0,0.0,-0.051221,-1.573703,-0.413745,-1.669847,-1.248820
1052,0.661329,0.0,0.0,0.194062,0.909370,0.265013,0.909370,0.434912
1151,-0.215675,0.0,0.0,-0.218413,0.483702,-0.003593,0.534003,-0.243106
242,0.168736,0.0,0.0,0.194062,-0.812546,-0.732503,-1.002675,-0.243106
1169,-0.066344,0.0,0.0,0.194062,-0.812546,-0.190462,-0.923211,-0.243106


Entrenamiento y evaluación

In [5]:
models = train_all_models(X_train_woe, y_train)

tabla_auc = evaluate_models(models, X_test_woe, y_test)

display(tabla_auc)

,Modelo,AUC
2,Logistic Regression,0.658487
1,XGBoost,0.638537
0,Random Forest,0.631790


Serialización del mejor modelo

In [6]:
mejor_nombre = tabla_auc.iloc[0]["Modelo"]
mejor_modelo = models[mejor_nombre]

metadata = {
    "model_name": mejor_nombre,
    "version": "1.0",
    "author": "teixeiradossantos",
    "dataset": "data/raw/bankloan.csv",
    "n_train_samples": int(len(y_train)),
    "n_test_samples": int(len(y_test)),
    "features_selected": features_sel,
    "hyperparameters": mejor_modelo.get_params(),
    "metrics": {
        "auc_test": round(
            auc_roc(mejor_modelo, X_test_woe, y_test),
            4
        ),
        "costo_total_test": int(
            costo_total(
                y_test.values,
                mejor_modelo.predict_proba(X_test_woe)[:, 1],
                umbral=0.5
            )
        ),
    },
}

save_model(
    mejor_modelo,
    path="../models/baseline_v1",
    metadata=metadata
)

print("Modelo serializado ✓")

Modelo serializado ✓
